<a href="https://colab.research.google.com/github/BialaStrzala/Sztuczna-Inteligencja-Paula-Grzebyk-21236/blob/main/SI_wstepne_przetwarzanie_danych.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paula Grzebyk

**Studium przypadków dotyczące audiobooków – wstępne przetwarzanie danych**

In [285]:
import tensorflow as tf
import numpy as np
from sklearn import preprocessing

In [286]:
raw_csv_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')
unscaled_inputs_all = raw_csv_data[:,1:-1]
targets_all = raw_csv_data[:,-1]

shuffled_indices = np.arange(unscaled_inputs_all.shape[0])
np.random.shuffle(shuffled_indices)

unscaled_inputs_all = unscaled_inputs_all[shuffled_indices]
targets_all = targets_all[shuffled_indices]

In [287]:
from sklearn.utils import validation
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []
#aby 0 i 1 bylo ok tyle samo
for i in range(targets_all.shape[0]):
  if targets_all[i] == 0:
    zero_targets_counter += 1
    if zero_targets_counter > num_one_targets:
      indices_to_remove.append(i)

#dane wejsciowe
unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
#targets
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

#normalizacja danych
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)


#przetasowanie 2
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

# wejsciowe
shuffled_inputs = scaled_inputs[shuffled_indices]
# targets
shuffled_targets = targets_equal_priors[shuffled_indices]


samples_count = shuffled_inputs.shape[0] #liczba probek
train_samples_count = int(0.8*samples_count) #trening
validation_samples_count = int(0.1*samples_count) #walidacja
test_samples_count = samples_count - train_samples_count - validation_samples_count #testowy

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]

print(np.sum(train_targets), train_samples_count, np.sum(train_targets)/train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets)/validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets)/test_samples_count)

np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

1805.0 3579 0.5043308186644314
209.0 447 0.46756152125279643
223.0 448 0.49776785714285715


In [288]:
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(np.float32)
train_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(np.float32)
validation_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(np.float32)
test_targets = npz['targets'].astype(np.int32)

In [289]:
input_size = 10
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [290]:
batch_size = 100
max_epochs = 100
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)

model.fit(train_inputs,
          train_targets,
          batch_size=batch_size,
          epochs=max_epochs,
          callbacks=[early_stopping],
          validation_data=(validation_inputs,validation_targets),
          verbose=1)

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6189 - loss: 0.6587 - val_accuracy: 0.7629 - val_loss: 0.5532
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7502 - loss: 0.5259 - val_accuracy: 0.7852 - val_loss: 0.4673
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7611 - loss: 0.4675 - val_accuracy: 0.7875 - val_loss: 0.4357
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7751 - loss: 0.4385 - val_accuracy: 0.7942 - val_loss: 0.4174
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7832 - loss: 0.4212 - val_accuracy: 0.8009 - val_loss: 0.4020
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7874 - loss: 0.4101 - val_accuracy: 0.7852 - val_loss: 0.4025
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7938 - loss: 0.3984 - val_accuracy: 0.8076 - val_loss: 0.3863
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7921 - loss: 0.3922 - val_accuracy: 0.7987 - v

In [291]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('\nTest loss: {0:.2f}. Test accuracy: {1:.2f}'.format(test_loss, test_accuracy*100.))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8304 - loss: 0.3552 

Test loss: 0.36. Test accuracy: 83.04


# Najlepsze wyniki

batch_size = 100

hidden_layer_size = 50

patience = 2

Test loss: 0.36. Test accuracy: 83.04
